# LoRA: Low-Rank Fine-Tuning

> A general base model is trained on broad data, then adapted to specific tasks with smaller datasets. Full fine-tuning a 7B model requires about 112 GB for fixed training state. LoRA keeps the base weight $W_0$ frozen and trains only a low-rank update $\Delta W=BA$.
>
> Matrix $A$ compresses the input to a small rank and $B$ projects it back to the output dimension. Together they approximate a full weight update.
>
> The adapter lifecycle is: select target Linear layers, inject LoRA branches, freeze the base, train adapters, then merge or save them separately.
>
> Dense models usually target Attention and FFN projections. MoE models must also distinguish experts from the router, while QLoRA additionally quantizes the frozen base.

For a $4096	imes4096$ Linear layer, a full update has about 16.78 million values. With rank $r=8$, $BA$ needs only $4096	imes8+8	imes4096=65,536$ trainable values. This chapter begins with the meaning of $\Delta W$ and then shows why the low-rank form reduces trainable state.


## 0. The Low-Rank Structure of Fine-Tuning Updates

Define the chapter's central quantity: the **weight update $\Delta W$**. Full fine-tuning changes a pretrained matrix from $W_0$ to $W_1$, so $\Delta W=W_1-W_0$ contains everything learned during adaptation.

If every element of $\Delta W$ carries independent information, the whole matrix must be stored. If most information lies in a few directions, a small number of basis components can reconstruct it.

A matrix is **low-rank** when it can be written as $BA$, where $B$ has shape $d_{out}	imes r$, $A$ has shape $r	imes d_{in}$, and $r\ll d$. LoRA's central assumption is that fine-tuning produces a low-rank $\Delta W$.


In [ ]:
# Keep all imports for this chapter in the first code cell
import copy
import math

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

torch.manual_seed(42)
print("This chapter follows two tracks: apply LoRA once to a dense base and once to an MoE base.")
print('Please replace the placeholder before running the assertion.')


In [ ]:
# Low-rank example: a 4x4 matrix appears to need 16 values, but eight are enough
u = torch.tensor([[1.0], [2.0], [3.0], [0.5]])    # 4x1 column vector
v = torch.tensor([[2.0, -1.0, 0.5, 3.0]])         # 1x4 row vector
M1 = u @ v                                        # Outer product: 4x4 but rank 1

print("Matrix produced by u @ v:")
print(M1)
print()
print("Matrix rank =", torch.linalg.matrix_rank(M1).item())
print("Number of matrix elements =", M1.numel())
print("Parameters needed to reconstruct it =", u.numel() + v.numel())
print()
print("Storing only u and v reconstructs the whole matrix; this is what low rank means.")
print("The two factors contain the structure instead of storing every matrix entry independently.")

# A rank-two matrix is the sum of two outer products
u2 = torch.tensor([[0.3], [-1.0], [0.8], [0.2]])
v2 = torch.tensor([[1.0, 0.5, -2.0, 0.7]])
M2 = u @ v + u2 @ v2
print()
print("Rank of a matrix formed by two outer products =", torch.linalg.matrix_rank(M2).item())
print("LoRA's product BA is this kind of sum: a larger r can combine more patterns.")


Is a learned $\Delta W$ actually low-rank? Two findings support the assumption. Aghajanyan et al. showed that constraining GPT-2 updates to a subspace of only hundreds or thousands of dimensions approached full fine-tuning performance. The LoRA paper also measured rapidly decaying singular values in GPT-3 updates, with most energy concentrated in a few leading directions.

This is intuitive: pretraining already supplies grammar, knowledge, and general reasoning. Adapting a dialogue task often changes a limited set of response patterns rather than rewriting every weight direction. LoRA therefore asks the optimizer to learn $A$ and $B$ directly and represents $\Delta W$ by their product.


## 1. Memory Cost of Full Fine-Tuning and Freezing

With AdamW and mixed precision, each trainable parameter occupies about 16 bytes:

| Component | Precision | Bytes per parameter |
|:--|:--|:--|
| Weight | FP16 | 2 |
| Gradient | FP16 | 2 |
| Adam first moment $m$ | FP32 | 4 |
| Adam second moment $v$ | FP32 | 4 |
| Master weight | FP32 | 4 |

Gradients and optimizer state account for 14 of those bytes. A frozen parameter participates in the forward pass but needs neither, so LoRA limits these expensive states to millions of adapter parameters instead of billions of base parameters.

The benefit is even larger for MoE models, whose many expert weights dominate total parameters. Freezing the experts avoids optimizer state for every one of them.


In [ ]:
# Calculate concrete memory costs for dense and MoE models under full fine-tuning, LoRA, and QLoRA
def training_memory(name, total_params, trainable_params, frozen_bytes=2):
    """Estimate fixed training-memory cost, excluding activations."""
    frozen = total_params - trainable_params
    # Frozen parameters store weights only; trainable ones need weights, gradients, m, v, and master weights
    total_bytes = frozen * frozen_bytes + trainable_params * 16
    print(f"{name:24s} {total_bytes / 1e9:8.1f} GB")
    return total_bytes


dense_7b = 7e9
moe_47b = 47e9                 # Total parameters in Mixtral 8x7B
dense_lora_ratio = 0.004       # Approximate all-linear trainable share with r=16
moe_lora_ratio = 0.002         # Larger MoE denominator gives a smaller percentage

print("=== Estimated fixed training memory, excluding activations ===")
training_memory("7B dense full fine-tuning", dense_7b, dense_7b)
training_memory("7B dense LoRA", dense_7b, dense_7b * dense_lora_ratio)
training_memory("47B MoE full fine-tuning", moe_47b, moe_47b)
training_memory("47B MoE LoRA", moe_47b, moe_47b * moe_lora_ratio)
training_memory("7B dense QLoRA", dense_7b, dense_7b * dense_lora_ratio, frozen_bytes=0.5)
print()
print("Key observation: gradients and optimizer states, rather than weights alone, dominate memory.")
print("Full fine-tuning a 47B MoE needs more than 700 GB, while LoRA needs about 95 GB;")
print("QLoRA then stores frozen weights in 4-bit form, cutting the requirement by more than half again.")


In [ ]:
# === Visualize estimated training memory on a log scale because the values span a wide range ===
cases = [
    ("7B dense\nfull", dense_7b, dense_7b, 2),
    ("7B dense\nLoRA", dense_7b, dense_7b * dense_lora_ratio, 2),
    ("47B MoE\nfull", moe_47b, moe_47b, 2),
    ("47B MoE\nLoRA", moe_47b, moe_47b * moe_lora_ratio, 2),
    ("7B dense\nQLoRA", dense_7b, dense_7b * dense_lora_ratio, 0.5),
]
# Use the same accounting as training_memory: frozen_bytes for frozen weights and 16 bytes for trainable ones
gbs = [((total - train) * fb + train * 16) / 1e9
       for _, total, train, fb in cases]

plt.figure(figsize=(9, 4))
bars = plt.bar([name for name, *_ in cases], gbs)
for bar, gb in zip(bars, gbs):
    plt.text(bar.get_x() + bar.get_width() / 2, gb * 1.15,
             f"{gb:.0f} GB", ha="center", fontsize=9)
plt.yscale("log")
plt.ylim(1, 3000)
plt.ylabel("Training memory (GB, log scale)")
plt.title("Fixed training memory by scheme (weights + grads + optimizer)")
plt.tight_layout()
plt.show()


## 2. LoRA Formula and Initialization

LoRA adds a branch beside a linear layer:

$$h=W_0x+rac{lpha}{r}BAx$$

- $W_0$: frozen pretrained weight of shape $d_{out}	imes d_{in}$
- $A$: trainable $r	imes d_{in}$ matrix, randomly initialized
- $B$: trainable $d_{out}	imes r$ matrix, initialized to zero
- $r$: rank, commonly 8, 16, or 32
- $lpha$: scaling coefficient; $lpha/r$ controls branch strength

The input is compressed to $r$ dimensions by $A$ and expanded by $B$. Initializing $B$ to zero makes $BA=0$, so the initial model exactly matches the base. $A$ and $B$ cannot both be zero: each matrix's gradient depends on the other, so both gradients would remain zero. Random $A$ supplies directions while zero $B$ preserves the initial output.


In [ ]:
# Hand-calculate one LoRA forward pass with d_in=4, d_out=4, r=2, and alpha=2
d_in, d_out, r, alpha = 4, 4, 2, 2

W0 = torch.tensor([
    [ 0.5, -0.3,  0.8, -0.2],
    [-0.4,  0.6, -0.1,  0.7],
    [ 0.3, -0.5, -0.6,  0.4],
    [-0.7,  0.2,  0.5, -0.8],
])
A = torch.randn(r, d_in) * 0.02      # Randomly initialized 2x4 matrix
B = torch.zeros(d_out, r)            # Zero-initialized 4x2 matrix
x = torch.tensor([1.0, 0.5, -0.5, -1.0])

h_base = W0 @ x                      # Original model output

z = A @ x                            # Step 1: 2x4 @ 4 reduces to r=2 dimensions
delta = (B @ z) * (alpha / r)        # Step 2: 4x2 @ 2 returns to four dimensions, then scales
h_lora = h_base + delta

print('"Input x:"', x.tolist())
print("Original output W0 @ x:", [round(t, 4) for t in h_base.tolist()])
print()
print("Step 1, A @ x, reduced to two dimensions:", [round(t, 4) for t in z.tolist()])
print("Step 2, B @ z, expanded back to four dimensions:", [round(t, 4) for t in (B @ z).tolist()])
print("Scale alpha / r =", alpha / r)
print()
print('"LoRA Output:"', [round(t, 4) for t in h_lora.tolist()])
print("Initially the bypass is exactly zero because B is zero.")
print("Training moves B away from zero, allowing the bypass to produce a correction.")


In [ ]:
# Counterexample: what happens if both A and B start at zero?
A0 = torch.zeros(2, 4, requires_grad=True)
B0 = torch.zeros(4, 2, requires_grad=True)
x_probe = torch.randn(4)

loss = (B0 @ (A0 @ x_probe)).pow(2).sum()
loss.backward()

print("Maximum absolute gradient of A:", A0.grad.abs().max().item())
print("Maximum absolute gradient of B:", B0.grad.abs().max().item())
print()
print("Both gradients vanish when both factors start at zero.")
print("LoRA therefore uses random A and zero B: the output starts at zero while gradients remain active.")


## 3. Implementing `LoraLinear`

`LoraLinear` wraps an existing `nn.Linear` and performs three jobs:

1. freeze the wrapped weight and bias;
2. hold trainable `lora_A` and `lora_B` parameters and compute $W_0x+(lpha/r)BAx$;
3. provide `merged_linear()` to return an ordinary `nn.Linear` with weight $W_0+(lpha/r)BA$.

Its public interface matches `nn.Linear`, allowing it to replace any linear layer during injection.


In [ ]:
class LoraLinear(nn.Module):
    """Attach a LoRA bypass to nn.Linear: y = W0 x + (alpha/r) * B(A(x))."""

    def __init__(self, linear, r=8, alpha=16):
        super().__init__()
        self.linear = linear
        self.in_features = linear.in_features
        self.out_features = linear.out_features
        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        # Freeze the original weight so gradients no longer update W0
        self.linear.weight.requires_grad = False
        if self.linear.bias is not None:
            self.linear.bias.requires_grad = False

        # Initialize A randomly and initialize B to all zeros
        self.lora_A = nn.Parameter(torch.randn(r, self.in_features) * 0.02)
        self.lora_B = nn.Parameter(torch.zeros(self.out_features, r))

    def forward(self, x):
        # Main path: frozen W0 x
        original = self.linear(x)
        # Bypass: reduce to r dimensions, expand back, and apply the scale
        bypass = (x @ self.lora_A.T) @ self.lora_B.T * self.scaling
        return original + bypass

    def merged_linear(self):
        """Merge the bypass into the weight and return a plain nn.Linear for zero-overhead inference."""
        merged = nn.Linear(
            self.in_features, self.out_features,
            bias=self.linear.bias is not None,
        )
        with torch.no_grad():
            delta_w = self.lora_B @ self.lora_A * self.scaling
            merged.weight.copy_(self.linear.weight + delta_w)
            if self.linear.bias is not None:
                merged.bias.copy_(self.linear.bias)
        return merged


print("LoraLinear is ready: freezing, bypass computation, and merging each have a clear method or attribute.")


In [ ]:
# Three checks: initial equivalence, only A and B trainable, and merge equivalence
torch.manual_seed(0)
base = nn.Linear(6, 4, bias=True)
layer = LoraLinear(base, r=2, alpha=4)
x = torch.randn(3, 6)

# Check 1: immediately after attaching LoRA, output matches the original layer exactly
with torch.no_grad():
    diff = (layer(x) - base(x)).abs().max().item()
print("Check 1, maximum initial difference from the original layer =", diff)
assert diff == 0.0

# Check 2: only lora_A and lora_B are trainable
trainable = [n for n, p in layer.named_parameters() if p.requires_grad]
print("Check 2, trainable parameters =", trainable)
assert trainable == ["lora_A", "lora_B"]

# Check 3: make B nonzero to simulate training, then compare before and after merging
with torch.no_grad():
    layer.lora_B.normal_(0, 0.1)
merged = layer.merged_linear()
with torch.no_grad():
    diff = (merged(x) - layer(x)).abs().max().item()
print("Check 3, maximum output difference before and after merging =", diff)
assert diff < 1e-6
print()
print("Key observation: merging preserves the result while removing the inference-time bypass.")


## 4. Base Model

The earlier `TinyCausalLM` had only two matrices to keep the loss lesson focused. LoRA needs named Attention and FFN projections, so we use a complete decoder with `W_Q`, `W_K`, `W_V`, `W_O`, `w_up`, and `w_down`.

The training pipeline is unchanged: next-token shifting, SFT loss only on assistant tokens using label `-100` elsewhere, padded batching, and a small AdamW training loop. Named projections will let the injection code select precise targets.


In [ ]:
def sinusoidal_pos(max_len, d_model):
    """Return a fixed sinusoidal position-encoding matrix of shape [max_len, d_model]."""
    pos = torch.arange(max_len).unsqueeze(1)
    div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)
    return pe


class CausalSelfAttention(nn.Module):
    """Multi-head causal self-attention with four named Linear projections for LoRA injection."""

    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0
        self.num_heads = num_heads
        self.d_head = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x):
        B, S, D = x.shape
        q = self.W_Q(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        k = self.W_K(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        v = self.W_V(x).view(B, S, self.num_heads, self.d_head).transpose(1, 2)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.d_head)
        causal = torch.triu(torch.ones(S, S, dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(causal, float("-inf"))
        attn = F.softmax(scores, dim=-1)
        out = (attn @ v).transpose(1, 2).reshape(B, S, D)
        return self.W_O(out)


class FFN(nn.Module):
    """Two-layer FFN: expand, apply GELU, then project back down."""

    def __init__(self, d_model, ffn_dim):
        super().__init__()
        self.w_up = nn.Linear(d_model, ffn_dim)
        self.w_down = nn.Linear(ffn_dim, d_model)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_up(x)))


class DenseBlock(nn.Module):
    """Pre-norm Transformer block with residual paths around Attention and FFN."""

    def __init__(self, d_model, num_heads, ffn_dim):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FFN(d_model, ffn_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniLM(nn.Module):
    """Tiny decoder-only language model with interchangeable dense or MoE blocks."""

    def __init__(self, vocab_size, blocks, d_model, max_len=32):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_encoding", sinusoidal_pos(max_len, d_model))
        self.blocks = nn.ModuleList(blocks)
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)

    def forward(self, input_ids, attention_mask=None, labels=None):
        """Return logits for token IDs and also compute loss when labels are provided."""
        S = input_ids.size(1)
        x = self.token_embedding(input_ids) + self.pos_encoding[:S]
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits.reshape(-1, logits.size(-1)),
                labels.reshape(-1),
                ignore_index=-100,
            )
        return {"loss": loss, "logits": logits}


print("MiniLM is ready: four named Attention projections and two named FFN layers are all Linear modules.")


In [ ]:
# Build SFT data with a tiny character vocabulary and eight conversations, following the training chapter
PAD_ID, BOS_ID, EOS_ID, ASST_ID = 0, 1, 2, 3

chars = "helohwaryuimistntodfngcsbupkwqvjaxz"
id_of = {"<pad>": PAD_ID, "<bos>": BOS_ID, "<eos>": EOS_ID, "<asst>": ASST_ID}
for ch in chars:
    id_of[ch] = len(id_of)
token_of = {i: t for t, i in id_of.items()}
VOCAB_SIZE = len(id_of)

conversations = [
    ("hello", "hello how may i help"),
    ("who are you", "i am a tiny assistant"),
    ("how is weather", "it is sunny go outside"),
    ("thanks", "you are welcome"),
    ("goodbye", "goodbye have a nice day"),
    ("can you code", "i can code a little"),
    ("what day is it", "today is monday"),
    ("are you human", "i am a model not a human"),
]


def encode(text):
    """Look up characters in the vocabulary and skip unknown ones for this teaching example."""
    return [id_of[ch] for ch in text if ch in id_of]


def build_sample(user_text, assistant_text):
    """Convert one conversation into shifted inputs and labels, masking all non-assistant targets."""
    user_ids = encode(user_text)
    asst_ids = encode(assistant_text)
    seq = [BOS_ID] + user_ids + [ASST_ID] + asst_ids + [EOS_ID]
    asst_pos = 1 + len(user_ids) + 1              # Index of the first assistant token
    labels = []
    for i in range(1, len(seq)):                  # labels[i-1] is the token predicted after seq[:i]
        labels.append(seq[i] if i >= asst_pos else -100)
    return {"input_ids": seq[:-1], "labels": labels}


print("Vocabulary size:", VOCAB_SIZE, " conversations:", len(conversations))


In [ ]:
# Print one sample's supervision: which positions contribute to loss?
sample = build_sample("can you code", "i can code a little")
tokens = [token_of[t] for t in sample["input_ids"]]

for i, (tok, lab) in enumerate(zip(tokens, sample["labels"])):
    if lab == -100:
        print(f"Position {i}: context {tokens[: i + 1]} -> ignored by loss")
    else:
        print(f"Position {i}: context {tokens[: i + 1]} -> predict {token_of[lab]}")

print()
print("Key observation: only the assistant response and its final <eos> token are supervised.")


In [ ]:
# Reuse simple_collate from the training chapter: pad inputs and fill label padding with -100
def simple_collate(features, pad_id=PAD_ID, ignore_index=-100):
    """Pad several samples into one batch containing input_ids, attention_mask, and labels."""
    max_len = max(len(f["input_ids"]) for f in features)
    batch_ids, batch_mask, batch_labels = [], [], []
    for f in features:
        pad = max_len - len(f["input_ids"])
        batch_ids.append(f["input_ids"] + [pad_id] * pad)
        batch_mask.append([1] * len(f["input_ids"]) + [0] * pad)
        batch_labels.append(f["labels"] + [ignore_index] * pad)
    return {
        "input_ids": torch.tensor(batch_ids),
        "attention_mask": torch.tensor(batch_mask),
        "labels": torch.tensor(batch_labels),
    }


class SFTDataset(Dataset):
    """Wrap the conversation list as a Dataset that returns one sample dictionary at a time."""

    def __init__(self, conversations):
        self.samples = [build_sample(u, a) for u, a in conversations]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


sft_dataset = SFTDataset(conversations)
sft_loader = DataLoader(sft_dataset, batch_size=4, shuffle=False, collate_fn=simple_collate)
first_batch = next(iter(sft_loader))

for name, value in first_batch.items():
    print(f"{name}: shape = {tuple(value.shape)}")
print()
print("Key observation: from this point on, batch shapes exactly match those in the training chapter.")


### Pretraining the Base Model

LoRA assumes a pretrained base. Applying SFT to a randomly initialized frozen embedding and language-model head would be unfair and unrealistic. We therefore pretrain the base on ordinary sentences with next-token loss over every token. This corpus differs from the SFT dialogue data, representing general pretraining followed by domain adaptation.


In [ ]:
def train_one_model(model, dataloader, epochs, lr):
    """Train one model using only parameters whose requires_grad flag is true."""
    trainable = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(trainable, lr=lr)
    history = []
    model.train()
    for epoch in range(epochs):
        total, steps = 0.0, 0
        for batch in dataloader:
            loss = model(**batch)["loss"]
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item()
            steps += 1
        history.append(total / steps)
    return history


# Pretraining corpus: ordinary sentences rather than SFT dialogues, simulating general base-model text
pretrain_sentences = [
    "it is sunny today go outside",
    "today is monday go outside and code",
    "who am i i am a tiny assistant",
    "hello tell me who you are",
    "today i write a little code",
    "goodbye have a nice day outside",
    "how is weather shall we go outside",
    "i am a model and i can code",
]


def pretrain_sample(text):
    """Create a pretraining sample in which the full sentence contributes to loss."""
    ids = [BOS_ID] + encode(text) + [EOS_ID]
    return {"input_ids": ids[:-1], "labels": ids[1:]}


class PretrainDataset(Dataset):
    """Wrap pretraining sentences as a Dataset."""

    def __init__(self, sentences):
        self.samples = [pretrain_sample(s) for s in sentences]

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        return self.samples[index]


pretrain_loader = DataLoader(
    PretrainDataset(pretrain_sentences),
    batch_size=4,
    shuffle=False,
    collate_fn=simple_collate,
)

# Build and pretrain the dense base
torch.manual_seed(42)
dense_base = MiniLM(
    VOCAB_SIZE,
    [DenseBlock(64, 4, ffn_dim=256) for _ in range(2)],
    d_model=64,
)
pretrain_history = train_one_model(dense_base, pretrain_loader, epochs=80, lr=1e-3)
print(f"Dense-base pretraining finished; loss fell from {pretrain_history[0]:.4f} to {pretrain_history[-1]:.4f}")
print("Key observation: the base learned common character patterns but has not seen the SFT conversations yet.")


## 5. Injecting LoRA and Freezing the Base

The main design choice is which layers receive adapters:

| Configuration | Targets | Origin |
|:--|:--|:--|
| Conservative | `W_Q` and `W_V` only | Original LoRA paper |
| Common | all Attention and FFN projections | QLoRA and later practice |
| Cautious extension | embedding / `lm_head` | Usually avoided because they strongly affect output semantics |

Adapters normally modify transformations of intermediate representations. Injection traverses the model and wraps matching `nn.Linear` modules. Afterward, every non-LoRA parameter—including LayerNorm and embeddings—must be frozen. “Train only LoRA” means that A and B are the only trainable parameters in the entire model.


In [ ]:
def inject_lora(model, target_names, r=8, alpha=16):
    """Replace matching nn.Linear children with LoraLinear and freeze every other parameter."""
    for name, module in model.named_modules():
        for child_name, child in module.named_children():
            if child_name in target_names and isinstance(child, nn.Linear):
                setattr(module, child_name, LoraLinear(child, r=r, alpha=alpha))
    # Freeze every parameter outside the target list; this is what makes training LoRA-only
    for p in model.parameters():
        p.requires_grad = False
    for n, p in model.named_parameters():
        if "lora_" in n:
            p.requires_grad = True
    return model


def param_audit(model, name=""):
    """Print total and trainable parameter counts and their ratio."""
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"[{name}] total {total:,}  trainable {trainable:,}  ({trainable / total:.2%})")
    return total, trainable


# Demonstrate the original paper's q,v target configuration
param_audit(dense_base, "pretrained dense base")
dense_lora = inject_lora(
    copy.deepcopy(dense_base), target_names={"W_Q", "W_V"}, r=8, alpha=16
)
param_audit(dense_lora, "after attaching LoRA")

print()
lora_names = [n for n, _ in dense_lora.named_parameters() if "lora_" in n]
print("LoRA tensors:", lora_names)
print("Key observation: only A and B on W_Q and W_V in each layer remain trainable.")
print("This tiny model gives about 4%; with the same setup, a real 7B model is around 0.1%.")


## 6. Fine-Tuning a Dense Base Model

Four models share the same pretrained initialization, data, and training loop:

- **Full fine-tuning** trains every parameter.
- **LoRA (all-linear)** adapts all Attention and FFN projections.
- **LoRA (q,v)** adapts only `W_Q` and `W_V`.
- **Frozen base** trains nothing and serves as a reference.

LoRA has less capacity, so its final loss need not equal full fine-tuning. Observe whether few parameters beat the frozen base and how all-linear differs from q,v.


In [ ]:
EPOCHS = 150
ALL_LINEAR = {"W_Q", "W_K", "W_V", "W_O", "w_up", "w_down"}

# Three models share one pretrained initialization and differ only in which parameters are trainable
full_model = copy.deepcopy(dense_base)
lora_model = inject_lora(copy.deepcopy(dense_base), ALL_LINEAR, r=8, alpha=16)
lora_qv_model = dense_lora           # Model with q,v LoRA attached above

with torch.no_grad():
    frozen_loss = dense_base(**first_batch)["loss"].item()
print(f"Frozen base loss on SFT data = {frozen_loss:.4f}, pretrained but unseen conversations")
print()

print("Starting training...")
history_full = train_one_model(full_model, sft_loader, EPOCHS, lr=1e-3)
print('"training LoRA（all-linear）..."')
history_lora = train_one_model(lora_model, sft_loader, EPOCHS, lr=3e-3)
print('"training LoRA（q,v）..."')
history_lora_qv = train_one_model(lora_qv_model, sft_loader, EPOCHS, lr=3e-3)

n_full = sum(p.numel() for p in full_model.parameters() if p.requires_grad)
n_lora = sum(p.numel() for p in lora_model.parameters() if p.requires_grad)
n_qv = sum(p.numel() for p in lora_qv_model.parameters() if p.requires_grad)
print(f"Full fine-tuning final loss: {history_full[-1]:.4f}, trainable {n_full:,}")
print(f"LoRA all-linear final loss: {history_lora[-1]:.4f}, trainable {n_lora:,}")
print(f"LoRA q,v final loss:        {history_lora_qv[-1]:.4f}, trainable {n_qv:,}")


In [ ]:
# Plot four loss curves: full fine-tuning, two LoRA variants, and the frozen base
plt.figure(figsize=(8, 4))
plt.plot(history_full, label=f"full fine-tuning ({n_full:,} trainable)")
plt.plot(history_lora, label=f"LoRA all-linear ({n_lora:,} trainable)")
plt.plot(history_lora_qv, label=f"LoRA q,v only ({n_qv:,} trainable)")
plt.axhline(frozen_loss, color="gray", linestyle="--", label=f"frozen base ({frozen_loss:.2f})")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("Dense base: LoRA vs full fine-tuning")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
def merge_lora(model):
    """Replace every LoraLinear in place with its merged plain nn.Linear."""
    for name, module in list(model.named_modules()):
        for child_name, child in list(module.named_children()):
            if isinstance(child, LoraLinear):
                setattr(module, child_name, child.merged_linear())
    return model


# Check 1: after LoRA training, every base weight remains bitwise unchanged
base_params = dict(dense_base.named_parameters())
changed = []
for name, p in lora_model.named_parameters():
    if "lora_" in name:
        continue
    base_name = name.replace(".linear.", ".")    # W_Q.linear.weight -> W_Q.weight
    if not torch.equal(base_params[base_name], p.detach()):
        changed.append(name)
assert changed == [], f"These base parameters should not change: {changed}"
print("Check 1 passed: every parameter except lora_A and lora_B exactly matches the pretrained base.")

# Check 2: model outputs agree before and after merging
with torch.no_grad():
    logits_bypass = lora_model(**first_batch)["logits"]
merged_model = merge_lora(copy.deepcopy(lora_model))
with torch.no_grad():
    logits_merged = merged_model(**first_batch)["logits"]
max_diff = (logits_bypass - logits_merged).abs().max().item()
print(f"Check 2, maximum logits difference before and after merging = {max_diff:.2e}")
assert max_diff < 1e-4
print("The merged model is an ordinary MiniLM with no bypass overhead during inference.")


## 7. MoE Base Model

MoE changes two details: experts replace the FFN and dominate parameter count, and a router is added. We reuse the earlier `MoELayer` but group tokens by expert instead of using three nested loops. The mathematics remains gate scores, top-$k$ selection, softmax over selected logits, and a weighted sum.

Experts use a named `Expert` class with `w_up` and `w_down`, rather than numeric `nn.Sequential` indices, so LoRA injection can match their names.


In [ ]:
class Expert(nn.Module):
    """One expert: a two-layer FFN with the same named projections as the dense FFN."""

    def __init__(self, d_model, expert_dim):
        super().__init__()
        self.w_up = nn.Linear(d_model, expert_dim)
        self.w_down = nn.Linear(expert_dim, d_model)

    def forward(self, x):
        return self.w_down(F.gelu(self.w_up(x)))


class MoELayer(nn.Module):
    """MoE layer: gate scores, top-k routing, and a weighted sum of expert outputs."""

    def __init__(self, d_model, num_experts=8, top_k=2, expert_dim=256):
        super().__init__()
        self.num_experts = num_experts
        self.top_k = top_k
        self.gate = nn.Linear(d_model, num_experts, bias=False)
        self.experts = nn.ModuleList([
            Expert(d_model, expert_dim) for _ in range(num_experts)
        ])

    def forward(self, x):
        B, S, D = x.shape
        gate_logits = self.gate(x)                          # [B, S, N]
        top_logits, top_idx = torch.topk(gate_logits, self.top_k, dim=-1)
        top_weights = F.softmax(top_logits, dim=-1)         # [B, S, k]

        flat_x = x.reshape(-1, D)
        flat_idx = top_idx.reshape(-1, self.top_k)
        flat_w = top_weights.reshape(-1, self.top_k)
        out = x.new_zeros(B * S, D)

        # Process tokens expert by expert, sending each expert only the tokens routed to it
        for e in range(self.num_experts):
            hit = flat_w * (flat_idx == e)      # Weight of matching slots; zero for misses
            w_e = hit.sum(dim=-1)               # Weight assigned to expert e for each token
            rows = w_e.nonzero().squeeze(-1)
            if rows.numel() == 0:
                continue
            expert_out = self.experts[e](flat_x[rows])
            out = out.index_add(0, rows, w_e[rows].unsqueeze(-1) * expert_out)

        return out.reshape(B, S, D)


class MoEBlock(nn.Module):
    """MoE version of DenseBlock: keep Attention and replace the FFN with MoELayer."""

    def __init__(self, d_model, num_heads, num_experts=8, top_k=2, expert_dim=256):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.moe = MoELayer(d_model, num_experts, top_k, expert_dim)

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.moe(self.ln2(x))
        return x


torch.manual_seed(42)
moe_base = MiniLM(
    VOCAB_SIZE,
    [MoEBlock(64, 4, num_experts=8, top_k=2, expert_dim=256) for _ in range(2)],
    d_model=64,
)
print("MoE base built: two layers, eight experts per layer, and top-2 routing.")

# As with the dense base, pretrain first and use the result as the fine-tuning starting point
pretrain_moe_history = train_one_model(moe_base, pretrain_loader, epochs=80, lr=1e-3)
print(f"MoE-base pretraining finished; loss fell from {pretrain_moe_history[0]:.4f} to {pretrain_moe_history[-1]:.4f}")


In [ ]:
# Audit where the MoE model stores most of its parameters
def param_breakdown(model):
    """Count parameters by Attention, gate, experts, and other modules."""
    groups = {"attention": 0, "router(gate)": 0, "experts(FFN)": 0, "embedding+lm_head": 0}
    for name, p in model.named_parameters():
        if ".attn." in name:
            groups["attention"] += p.numel()
        elif ".moe.gate." in name:
            groups["router(gate)"] += p.numel()
        elif ".moe.experts." in name:
            groups["experts(FFN)"] += p.numel()
        else:
            groups["embedding+lm_head"] += p.numel()
    total = sum(groups.values())
    for k, v in groups.items():
        print(f"{k:22s} {v:>10,}  ({v / total:.1%})")
    print(f"{'Total':22s} {total:>10,}")
    return groups


param_breakdown(moe_base)
print()
print("Key observation: expert matrices account for most MoE parameters.")
print("Full fine-tuning expands optimizer state in the same proportions, determining where MoE LoRA matters most.")


In [ ]:
# === Visualize where the MoE model stores most parameters ===
groups = param_breakdown(moe_base)

plt.figure(figsize=(8, 4))
bars = plt.bar(groups.keys(), [v / 1e6 for v in groups.values()])
# Label each bar with its share so it matches the printed counts above
total = sum(groups.values())
for bar, v in zip(bars, groups.values()):
    plt.text(bar.get_x() + bar.get_width() / 2,
             bar.get_height() + 0.01, f"{v / total:.1%}",
             ha="center", fontsize=9)
plt.ylabel("Parameters (millions)")
plt.title("Parameter breakdown of the MoE base model")
plt.tight_layout()
plt.show()


## 8. LoRA Injection Strategy for MoE

MoE introduces another decision: whether to adapt experts or the router.

| Location | Parameter share | Reason to adapt | Reason not to adapt |
|:--|:--|:--|:--|
| Attention projections | small | common effective targets | cannot modify expert knowledge |
| Expert projections | large | FFNs store much domain knowledge | one adapter pair per expert |
| Router gate | tiny | — | changes token allocation and may cause imbalance |

Common choices are Attention-only or Attention plus experts. The router is normally frozen: it has few parameters, while changing its allocation can destabilize expert load.

Each expert needs its own A and B. Gradients follow routing, so only experts selected by tokens in a step receive adapter gradients. An underused expert's LoRA may therefore receive little training.


In [ ]:
# Attach LoRA to Attention and every expert in the MoE base, leaving the gate frozen
moe_lora = inject_lora(
    copy.deepcopy(moe_base),
    target_names={"W_Q", "W_V", "w_up", "w_down"},
    r=8,
    alpha=16,
)
param_audit(moe_base, "pretrained MoE base")
param_audit(moe_lora, "MoE + LoRA(attn+experts)")

n_adapters = len([n for n, _ in moe_lora.named_parameters() if "lora_" in n])
print()
print(f"There are {n_adapters} LoRA tensors: W_Q and W_V plus 8 experts x 2 matrices per layer,")
print("for 36 A,B pairs across two layers. Experts account for 32 pairs, so they dominate adapter cost.")
print("Key observation: targeting all expert projections is much more expensive than targeting Attention alone.")


In [ ]:
# MoE experiment: LoRA versus full fine-tuning with the same initialization and data
EPOCHS_MOE = 120

moe_full = copy.deepcopy(moe_base)

with torch.no_grad():
    frozen_moe_loss = moe_base(**first_batch)["loss"].item()
print(f"Frozen MoE base loss on SFT data = {frozen_moe_loss:.4f}")
print()

print("Training full MoE fine-tuning ...")
history_moe_full = train_one_model(moe_full, sft_loader, EPOCHS_MOE, lr=1e-3)
print('"training MoE LoRA ..."')
history_moe_lora = train_one_model(moe_lora, sft_loader, EPOCHS_MOE, lr=3e-3)

n_moe_full = sum(p.numel() for p in moe_full.parameters() if p.requires_grad)
n_moe_lora = sum(p.numel() for p in moe_lora.parameters() if p.requires_grad)
print(f"Full MoE fine-tuning final loss: {history_moe_full[-1]:.4f}, trainable {n_moe_full:,}")
print(f"MoE LoRA final loss:             {history_moe_lora[-1]:.4f}, trainable {n_moe_lora:,}")


In [ ]:
# Plot the loss curve comparison
plt.figure(figsize=(8, 4))
plt.plot(history_moe_full, label=f"full fine-tuning ({n_moe_full:,} trainable)")
plt.plot(history_moe_lora, label=f"LoRA on attn+experts ({n_moe_lora:,} trainable)")
plt.axhline(frozen_moe_loss, color="gray", linestyle="--", label=f"frozen base ({frozen_moe_loss:.2f})")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy loss")
plt.title("MoE base: LoRA vs full fine-tuning")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Examine routing stability at two levels: gate weights and actual top-k choices
def probe_routing(model, batch):
    """Run to the first MoE layer and return top-k expert indices without training."""
    with torch.no_grad():
        S = batch["input_ids"].size(1)
        x = model.token_embedding(batch["input_ids"]) + model.pos_encoding[:S]
        block = model.blocks[0]
        x = x + block.attn(block.ln1(x))
        gate_logits = block.moe.gate(block.ln2(x))
        _, top_idx = torch.topk(gate_logits, block.moe.top_k, dim=-1)
    return top_idx


# Level 1: gate weights. LoRA does not target the gate, so they should remain bitwise unchanged
gate_base = moe_base.blocks[0].moe.gate.weight
gate_lora = moe_lora.blocks[0].moe.gate.weight
assert torch.equal(gate_base, gate_lora.detach()), "The frozen gate weights should not change"
print("Check passed: gate weights are bitwise unchanged after LoRA; its scoring rule was not trained.")

# Level 2: actual routing also depends on upstream Attention outputs
# Attention has LoRA adapters, so changed representations can move a few routing choices
routing_base = probe_routing(moe_base, first_batch)
routing_after_lora = probe_routing(moe_lora, first_batch)
routing_after_full = probe_routing(moe_full, first_batch)

gate_drift_full = (gate_base - moe_full.blocks[0].moe.gate.weight).norm().item()
keep_lora = (routing_base == routing_after_lora).float().mean().item()
keep_full = (routing_base == routing_after_full).float().mean().item()
print(f"Full fine-tuning changes gate weights themselves, norm {gate_drift_full:.3f}; LoRA change is zero")
print(f"Share of routing positions unchanged after LoRA:        {keep_lora:.1%}")
print(f"Share of routing positions unchanged after full tuning: {keep_full:.1%}")
print()
usage = torch.bincount(routing_after_lora.flatten(), minlength=8).float()
usage = usage / usage.sum()
print("Expert-use distribution after LoRA:", [f"{u:.0%}" for u in usage.tolist()])
print()
print("Key observation: omitting the gate from LoRA freezes the scoring rule exactly.")
print("Routing can still move slightly because upstream Attention adapters change representations.")
print("MoE plus LoRA therefore still requires monitoring expert utilization just as ordinary MoE training does.")


In [ ]:
# === Visualize routing stability under LoRA and full fine-tuning ===
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Left: share of positions whose routes remain unchanged after training
axes[0].bar(["LoRA", "full fine-tuning"], [keep_lora, keep_full])
axes[0].set_ylim(0, 1.05)
axes[0].set_ylabel("Fraction of unchanged top-k routing")
axes[0].set_title("Routing preserved vs. before training")

# Right: utilization of eight experts after LoRA, revealing any strong imbalance
axes[1].bar(range(8), usage.tolist())
axes[1].set_xlabel("Expert id")
axes[1].set_ylabel("Usage share")
axes[1].set_title("Expert usage after LoRA training")

plt.tight_layout()
plt.show()


## 9. Merging, Saving, and Switching LoRA

After training, adapters have three common uses.

**Merged deployment.** Replace each adapted matrix by $W_0+(lpha/r)BA$. MoE experts merge independently, while the gate remains unchanged.

**Adapter-only checkpoints.** Save only the small A and B tensors. One base can share separate customer-service, coding, and writing adapters without duplicating base weights.

**Runtime switching.** Keep one unmerged base resident and select adapters per request. Inference systems such as vLLM support multiple concurrent LoRA adapters.


In [ ]:
# Merge the MoE model and measure how small an adapter-only checkpoint is
with torch.no_grad():
    logits_bypass = moe_lora(**first_batch)["logits"]
moe_merged = merge_lora(copy.deepcopy(moe_lora))
with torch.no_grad():
    logits_merged = moe_merged(**first_batch)["logits"]
diff = (logits_bypass - logits_merged).abs().max().item()
print(f"Maximum MoE logits difference before and after merging = {diff:.2e}")
assert diff < 1e-4
print("The merged MoE has exactly the base architecture and can be deployed directly.")
print()

adapter_sd = {k: v for k, v in moe_lora.state_dict().items() if "lora_" in k}
n_adapter = sum(v.numel() for v in adapter_sd.values())
n_total = sum(p.numel() for p in moe_lora.parameters())
print(f"Number of adapter tensors: {len(adapter_sd)}")
print(f"Adapter parameters:        {n_adapter:,}, or {n_adapter / n_total:.2%} of the model")
print()
print(f"Key observation: adapters use {n_adapter / n_total:.0%} of this tiny model because its matrices are small;")
print("the same configuration is around 0.1% on a real large model, so one base can serve many tasks.")


In [ ]:
# === Visualize a full checkpoint versus an adapter-only checkpoint ===
# Convert both to MB assuming FP16 storage at two bytes per parameter
sizes_mb = [(n_total * 2) / 1e6, (n_adapter * 2) / 1e6]

plt.figure(figsize=(7, 3.5))
bars = plt.bar(["full checkpoint", "adapter only"], sizes_mb)
for bar, mb in zip(bars, sizes_mb):
    plt.text(bar.get_x() + bar.get_width() / 2, mb * 1.15,
             f"{mb:.1f} MB", ha="center", fontsize=9)
plt.yscale("log")
plt.ylabel("Checkpoint size (MB, log scale)")
plt.title("What you need to save after LoRA training")
plt.tight_layout()
plt.show()


## 10. QLoRA and Practical Settings

LoRA reduces trainable state, but FP16 base weights still require 14 GB for a 7B model. QLoRA stores the frozen base in 4-bit NF4, dequantizes for forward computation, and sends gradients only to LoRA parameters.

| Technique | Purpose |
|:--|:--|
| NF4 | 16 quantization levels designed for normally distributed weights |
| Double Quantization | quantizes the quantization constants themselves |
| Paged Optimizer | pages optimizer state through unified memory to smooth peaks |

Freezing and quantization compose naturally because the base is never updated. NF4 is examined in the quantization chapter.


In [ ]:
# Compare fixed memory for three approaches on a 7B base with all-linear r=16, about 0.4% trainable
P = 7e9
adapter = P * 0.004

rows = [
    ("full fine-tuning", P * 16),
    ("LoRA (FP16 base)", (P - adapter) * 2 + adapter * 16),
    ("QLoRA (NF4 base)", (P - adapter) * 0.5 + adapter * 16),
]
print(f"{'Approach':<20}{'Fixed memory'}")
for name, b in rows:
    print(f"{name:<20}{b / 1e9:6.1f} GB")
print()
print("Key observation: LoRA removes most gradient and optimizer-state cost.")
print("Quantization then reduces the storage precision of the frozen weights.")


In [ ]:
# === Visualize the memory comparison among the three approaches ===
names = [name for name, _ in rows]
gbs = [b / 1e9 for _, b in rows]

plt.figure(figsize=(8, 4))
bars = plt.bar(names, gbs)
for bar, gb in zip(bars, gbs):
    plt.text(bar.get_x() + bar.get_width() / 2, gb + 1.5,
             f"{gb:.1f} GB", ha="center", fontsize=9)
plt.ylabel("GPU memory (GB)")
plt.title("Memory footprint on a 7B base model")
plt.tight_layout()
plt.show()


### Hyperparameter Quick Reference

| Question | Common practice |
|:--|:--|
| Rank $r$ | start with 8 or 16; use 32 or 64 for more complex tasks |
| Alpha | often $2r$; the effective scale is $lpha/r$ |
| Target layers | all-linear is robust; q,v is a useful baseline |
| Learning rate | roughly $10^{-4}$ to $3	imes10^{-4}$ |
| Dropout | 0.05–0.25 for small datasets; often 0 with ample data |
| Epochs | dozens can be reasonable for small datasets; use validation early stopping |

LoRA tolerates a larger learning rate because the frozen base cannot be directly damaged; updates are confined to small branches. Its capacity is still lower than full fine-tuning. For large distribution shifts, such as continued pretraining in a new language, compare both methods on a small run before choosing.


## Summary (Checklist)

1. ✅ Fine-tuning learns $\Delta W=W_1-W_0$; low-rank adaptation uses $\Delta Wpprox BA$.
2. ✅ Full AdamW fine-tuning needs about 16 bytes per parameter; LoRA confines expensive state to adapters.
3. ✅ Random A and zero B preserve the initial base output without deadlocking gradients.
4. ✅ `LoraLinear` matches `nn.Linear`; after injection, freeze every other parameter.
5. ✅ More target layers provide more adaptation capacity.
6. ✅ Merging $W'=W_0+(lpha/r)BA$ restores the base architecture with no adapter inference overhead.
7. ✅ Experts dominate MoE parameters, so adapting them can be valuable.
8. ✅ Adapt Attention and optionally experts, but normally freeze the router gate.
9. ✅ Each expert has its own adapter and receives gradients only when routed tokens use it.
10. ✅ Adapter-only checkpoints let one base support many tasks.
11. ✅ QLoRA combines LoRA with a 4-bit NF4 base.
12. ✅ LoRA often uses a higher learning rate but has less capacity than full fine-tuning.

In one sentence: LoRA reparameterizes fine-tuning as learning two small matrices beside a frozen base; dense and MoE models use the same mechanism but choose different target layers.


## Exercises

> You can ask AI to help explain ideas or break down steps, but it is not recommended to let AI "do the whole problem" for you.

**Exercise 1: LoRA Parameter Count**

For a $4096	imes4096$ linear layer with rank $r=16$, compute the original parameter count, the additional A+B parameters, and their percentage of the original.

Hint: LoRA parameters equal $r d_{in}+d_{out}r$.


In [ ]:
# Exercise 1: count LoRA parameters
d_out, d_in = 4096, 4096
r = 16

# TODO: parameter count of the original weight
original_params = None

# TODO: parameters added by LoRA, A plus B
lora_params = None

# TODO: ratio of added parameters to original parameters
ratio = None

# Uncomment the checks below after filling the blanks
# assert original_params == d_out * d_in
# assert lora_params == r * d_in + d_out * r
# assert abs(ratio - lora_params / original_params) < 1e-9
# print(f"Exercise 1 passed: original {original_params:,}, LoRA adds {lora_params:,}, ratio {ratio:.2%}")
# print("   A 16M-parameter matrix needs only about 131K bypass parameters.")


**Exercise 2: Merge Weights by Hand**

$$W_0=egin{pmatrix}1&0\0&1\end{pmatrix},\quad
B=egin{pmatrix}0.6\0.2\end{pmatrix},\quad
A=egin{pmatrix}0.5&-1.0\end{pmatrix},\quad lpha/r=1$$

Compute $W'=W_0+(lpha/r)BA$.

Hint: $BA$ is a $2	imes1$ by $1	imes2$ outer product.


In [ ]:
# Exercise 2: hand-calculate weight merging
import torch

W0 = torch.tensor([[1.0, 0.0], [0.0, 1.0]])
B = torch.tensor([[0.6], [0.2]])
A = torch.tensor([[0.5, -1.0]])
scaling = 1.0

# TODO: compute delta = B @ A, then W_merged = W0 + scaling * delta
W_merged = None

# Uncomment the checks below after filling the blanks
# expected = W0 + scaling * (B @ A)
# assert torch.allclose(W_merged, expected, atol=1e-6)
# print(f"Exercise 2 passed: W merged = {W_merged.tolist()}")
# print("   The merged shape matches W0, so inference uses one ordinary matrix multiplication.")


**Exercise 3: LoRA Parameters for MoE Experts**

A Mixtral-style MoE layer has eight experts, each with a $14336	imes4096$ `w_up`; Attention has one $4096	imes4096$ `W_Q`. With $r=16$, compute the total LoRA parameters for all eight expert matrices, the parameters for `W_Q`, and their ratio.

Hint: one matrix needs $r(d_{in}+d_{out})$ parameters; the expert count determines how many copies exist.


In [ ]:
# Exercise 3: count LoRA parameters on MoE experts
num_experts = 8
d_model = 4096
expert_dim = 14336
r = 16

# TODO: total LoRA parameters for w_up on all eight experts
experts_lora_params = None

# TODO: LoRA parameters for W_Q alone
attn_lora_params = None

# TODO: ratio of the former to the latter
times = None

# Uncomment the checks below after filling the blanks
# assert experts_lora_params == num_experts * r * (d_model + expert_dim)
# assert attn_lora_params == r * (d_model + d_model)
# assert abs(times - experts_lora_params / attn_lora_params) < 1e-9
# print(f"Exercise 3 passed: experts {experts_lora_params:,}, W_Q {attn_lora_params:,}, ratio {times:.0f}x")
# print("   Expert matrices are wider and more numerous, so they dominate adapter cost.")


## References

- Hu et al., [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685), 2021 — original LoRA paper
- Aghajanyan et al., [Intrinsic Dimensionality Explains the Effectiveness of Language Model Fine-Tuning](https://arxiv.org/abs/2012.13255), 2020 — evidence for low-dimensional fine-tuning subspaces
- Dettmers et al., [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314), 2023 — NF4 base quantization with LoRA
- Biderman et al., [LoRA Learns Less and Forgets Less](https://arxiv.org/abs/2405.09673), 2024 — capacity comparison with full fine-tuning
- Liu et al., [DoRA: Weight-Decomposed Low-Rank Adaptation](https://arxiv.org/abs/2402.09353), 2024 — decomposes magnitude and direction before low-rank adaptation
